# BOI v1.20.1 — GPU exact-WJ over probed buckets (no nested-LSH filter)

Same banded BOI + shared GPU encoding CSR as v1.20, but candidate generation is changed:
instead of nested-LSH on the 100-dim signatures, we take **all members of the probed
buckets** and score them with **exact Weighted Jaccard on the real encodings** (reading the
shared CSR via member_ids — no per-bucket copies). Banding only selects buckets; exact WJ
does the rest. Goal: v1.11-level recall (~0.99) at GPU speed, with no vector duplication.

Compare against v1.20 (nested-LSH, R@500=0.947) and v1.11 dense (0.99, 26 QPS).

In [1]:
# ============================================================
# CONFIG — all tunable knobs live here
# ============================================================
import os

# ── Dataset ──────────────────────────────────────────────────
ENCODING_DIR         = "/raid/ssEncodingData/encoding/pk-real0.002"
ENCODING_FILE_PREFIX = "real_"
GROUND_TRUTH_DIR     = "/raid/ssEncodingData/warehouse/pk-query-187019"

TRAIN_RATIO          = 0.80
MAX_QUERIES          = None
ZERO_TOL             = 1e-12

# ── Shared cache (encodings, GT, flat arrays, signatures) ────
_dataset_key     = os.path.basename(ENCODING_DIR.rstrip("/"))
SHARED_CACHE_DIR = (
    f"/raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/{_dataset_key}/"
)

# ── Signatures ───────────────────────────────────────────────
L        = 100
WMH_SEED = 42

# ── Banding ──────────────────────────────────────────────────
B = 50
R = 2

# ── Rebalance ────────────────────────────────────────────────
REB_MERGE_SMALL = 0
REB_SPLIT_LARGE = 3000

# ── Nested LSH build ─────────────────────────────────────────
BUCKET_INDEX_MIN_SIZE = 1000
BUILD_WORKERS         = 60
nested_L2             = 40
nested_R2             = 2

# ── Bucket index cache (shared with v1.17) ────────────────────
_INDEX_CACHE_KEY = (
    f"L{L}_B{B}_R{R}"
    f"_minSz{BUCKET_INDEX_MIN_SIZE}"
    f"_rebSplit{REB_SPLIT_LARGE}"
    f"_nested_lsh"
    f"_nL{nested_L2}_nR{nested_R2}"
    f"_sigvec"
)
BUCKET_CACHE_DIR = os.path.join(
    "/raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k",
    _dataset_key,
    f"bucket_indexes_{_INDEX_CACHE_KEY}",
)

# ── Query ─────────────────────────────────────────────────────
FAST_TOPK        = 500
FAST_ALPHA_MULT  = 0.0    # no collision threshold — keep every probed-bucket member
FAST_EPSILON     = 200000 # effectively no cap — exact WJ ranks all candidates
MAX_BANDS_TO_PROBE = 40

# ── GPU ───────────────────────────────────────────────────────
EVAL_BATCH_SIZE  = 200   # candidate sets are larger here (full buckets) → smaller batch

# ── Encoding load ─────────────────────────────────────────────
ENC_LOAD_WORKERS = 32

print("Config ready.")
print(f"  dataset_key      : {_dataset_key}")
print(f"  SHARED_CACHE_DIR : {SHARED_CACHE_DIR}")
print(f"  BUCKET_CACHE_DIR : {BUCKET_CACHE_DIR}")


Config ready.
  dataset_key      : pk-real0.002
  SHARED_CACHE_DIR : /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/
  BUCKET_CACHE_DIR : /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/bucket_indexes_L100_B50_R2_minSz1000_rebSplit3000_nested_lsh_nL40_nR2_sigvec


In [2]:
import sys
sys.path.insert(0, "/raid/ruban/boi_multi_index")
import importlib
import boi_pipeline as bp
importlib.reload(bp)

import numpy as np, gc, os, time
import cupy as cp

bp.release_stale_shm()
bp.print_memory_stats("After startup")


Released shm: boi_ids_flat
Released shm: boi_vals_flat
Released shm: boi_offsets
Released shm: boi_sig_all
[After startup] RSS=0.30 GB | Avail=1455.93/2015.68 GB | Used=27.8%


In [3]:
encodings, DATA_END, QUERY_START, QUERY_END, POLY_COUNT = bp.load_encodings_cached(
    encoding_dir=ENCODING_DIR,
    cache_dir=SHARED_CACHE_DIR,
    train_ratio=TRAIN_RATIO,
    max_queries=MAX_QUERIES,
    file_prefix=ENCODING_FILE_PREFIX,
    zero_tol=ZERO_TOL,
    num_workers=ENC_LOAD_WORKERS,
)

gt_neighbors, gt_offsets, gt_qids, gt_qid_map = bp.load_ground_truth_cached(
    gt_dir=GROUND_TRUTH_DIR,
    cache_dir=SHARED_CACHE_DIR,
)

D = max(int(ids.max()) for ids, _ in encodings if ids.size > 0) + 1
print(f"D={D:,}")


[Cache hit] Loading encodings from /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/encodings_payload.pkl
Cached encodings loaded in 4.68s
POLY_COUNT=233,773 | DB=[0,187019) | Q=[187019,233773)
[After encoding load] RSS=9.22 GB | Avail=1446.96/2015.68 GB | Used=28.2%
[Cache hit] Loading GT from /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/gt_csr.pkl
D=18,219


In [4]:
ids_flat, vals_flat, offsets = bp.build_flat_encoding_arrays_cached(
    encodings=encodings,
    cache_dir=SHARED_CACHE_DIR,
)

sig_all = bp.compute_signatures_cached(
    encodings=encodings,
    ids_flat=ids_flat, vals_flat=vals_flat, offsets=offsets,
    L=L, seed=WMH_SEED,
    cache_dir=SHARED_CACHE_DIR,
)

del encodings; gc.collect()
bp.print_memory_stats("After signature compute")

SHM_META, _SHM_REGISTRY = bp.setup_shared_memory(ids_flat, vals_flat, offsets, sig_all)


[Cache hit] Loading flat arrays from /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/flat_encoding_arrays.pkl
[Cache hit] WMH params from /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/wmh_params_L100.pkl
Numba WMH warmup...
Warmup done.
[Cache hit] Signatures from /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/sig_all_L100.npy
sig_all: shape=(233773, 100) dtype=float32
[After signatures] RSS=18.72 GB | Avail=1437.47/2015.68 GB | Used=28.7%
[After signature compute] RSS=18.66 GB | Avail=1437.52/2015.68 GB | Used=28.7%
Moving arrays to shared memory...
Shared memory ready in 3.62s
[After shared memory setup] RSS=27.30 GB | Avail=1428.86/2015.68 GB | Used=29.1%


In [5]:
banding = bp.Banding(B=B, R=R)
assert L == B * R

sig_data = sig_all[:DATA_END]

part_map_raw, _ = bp.build_partition_map(sig_data, banding)
part_map, rebalance_stats = bp.rebalance_part_map(
    part_map_raw, sig_data, R=banding.R,
    merge_small_threshold=REB_MERGE_SMALL,
    split_large_threshold=REB_SPLIT_LARGE,
)
del part_map_raw; gc.collect()
print("Rebalance stats:")
for k, v in rebalance_stats.items():
    print(f"  {k}: {v}")


Building band partitions: 100%|██████████| 187019/187019 [00:14<00:00, 13161.25it/s]


Rebalance stats:
  total_buckets: 207902
  mean_bucket_size: 44.97768179238295
  median_bucket_size: 2.0
  max_bucket_size: 34095
  p90_bucket_size: 35.0


In [6]:
# ── Nested LSH build worker (shared with v1.17) ───────────────
from collections import defaultdict

def _build_one_bucket_nested_lsh(args):
    bucket_key, member_ids = args
    G  = bp._BUCKET_BUILD_GLOBALS
    L2 = G["nested_L2"]; R2 = G["nested_R2"]; B2 = L2 // R2

    X_bucket  = G["sig_all"][member_ids]
    inner_map = defaultdict(list)
    for local_idx in range(len(member_ids)):
        sig = X_bucket[local_idx]
        for b in range(B2):
            inner_map[(b, tuple(sig[b*R2 : b*R2+R2].tolist()))].append(local_idx)

    return {
        "bucket_key": bucket_key,
        "member_ids": member_ids,
        "inner_map":  {k: np.asarray(v, dtype=np.int32) for k, v in inner_map.items()},
        "use_index":  True,
        "index_type": "nested_lsh",
        "worker_elapsed_sec": 0.0,
    }


In [7]:
bucket_indexes, split_index, build_stats = bp.build_or_load_bucket_indexes(
    part_map=part_map,
    worker_fn=_build_one_bucket_nested_lsh,
    shm_meta=SHM_META,
    cache_dir=BUCKET_CACHE_DIR,
    min_bucket_size=BUCKET_INDEX_MIN_SIZE,
    build_workers=BUILD_WORKERS,
    extra_worker_kwargs={"nested_L2": nested_L2, "nested_R2": nested_R2},
    local_hnsw_efs=400,
)
print("Build stats:")
for k, v in build_stats.items():
    print(f"  {k}: {v}")


[Cache hit] Loading bucket indexes from /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/bucket_indexes_L100_B50_R2_minSz1000_rebSplit3000_nested_lsh_nL40_nR2_sigvec


Loading bucket indexes: 100%|██████████| 207902/207902 [00:00<00:00, 589586.41it/s]


Loaded 207,902 buckets (2,171 indexed, 205,731 exact)
Build stats:
  total_buckets: 207902
  indexed_buckets: 2171
  exact_only_buckets: 205731
  total_members_indexed: 5149371
  mean_bucket_size: 44.97768179238295
  median_bucket_size: 2.0
  max_bucket_size: 34095
  build_time_sec: 13.490703344345093
  avg_worker_build_sec: 0.0
  avg_reload_sec: 8.725175545655984e-06


In [8]:
# ============================================================
# Cell 1 — GPU CSR Index Build
# Replaces: part_map, bucket_indexes, split_index
# ============================================================
import cupy as cp
import numpy as np
import time

# ---- Hash function: (band_idx, int64, int64) -> uint64 ----
# Polynomial hash — collision rate negligible at this scale
HASH_P1 = np.int64(2654435761)   # Knuth multiplicative
HASH_P2 = np.int64(805459861)
HASH_P3 = np.int64(3474701543)

def hash_band_key(band_idx, v0, v1):
    """Map (band_idx, v0, v1) -> uint64 scalar."""
    h = np.int64(band_idx) * HASH_P1
    h = h ^ (np.int64(v0) * HASH_P2)
    h = h ^ (np.int64(v1) * HASH_P3)
    return int(h & np.int64(0x7FFFFFFFFFFFFFFF))  # keep positive

# ---- Step 1: Assign integer IDs to all unique bucket keys ----
print("Step 1: Assigning bucket IDs...")
t0 = time.time()

bucket_key_to_id = {}   # (band_idx, v0, v1) -> int bucket_id
bucket_id_to_key = []   # bucket_id -> original key (for lookup)

for key in part_map.keys():
    band_idx = key[0]
    # Handle split buckets (4-tuple) and normal buckets (2-tuple)
    if len(key) == 2:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1)
    elif len(key) == 4:
        # SPLIT bucket: (band_idx, (v0,v1), 'SPLIT', j)
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1, key[3])  # include split index
    else:
        hk = key

    if hk not in bucket_key_to_id:
        bucket_key_to_id[hk] = len(bucket_id_to_key)
        bucket_id_to_key.append(key)

NUM_BUCKETS = len(bucket_key_to_id)
print(f"  Unique buckets: {NUM_BUCKETS:,}  ({time.time()-t0:.2f}s)")

# ---- Step 2: Build outer CSR (bucket -> member poly IDs) ----
print("Step 2: Building outer CSR...")
t0 = time.time()

# Count members per bucket
outer_counts = np.zeros(NUM_BUCKETS, dtype=np.int32)
for key, members in part_map.items():
    band_idx = key[0]
    if len(key) == 2:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1)
    elif len(key) == 4:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1, key[3])
    else:
        hk = key
    bid = bucket_key_to_id[hk]
    outer_counts[bid] = len(members)

# CSR row pointers
outer_offsets = np.zeros(NUM_BUCKETS + 1, dtype=np.int32)
outer_offsets[1:] = np.cumsum(outer_counts)
total_outer = int(outer_offsets[-1])

# CSR column indices (member poly IDs)
outer_members = np.empty(total_outer, dtype=np.int32)
for key, members in part_map.items():
    band_idx = key[0]
    if len(key) == 2:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1)
    elif len(key) == 4:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1, key[3])
    else:
        hk = key
    bid = bucket_key_to_id[hk]
    s = outer_offsets[bid]
    e = outer_offsets[bid + 1]
    outer_members[s:e] = np.asarray(members, dtype=np.int32)

print(f"  Outer CSR: {NUM_BUCKETS:,} buckets, {total_outer:,} assignments ({time.time()-t0:.2f}s)")

# ---- Step 3: Build per-bucket metadata arrays ----
print("Step 3: Building bucket metadata...")
t0 = time.time()

# is_large[bid] = 1 if this bucket has a nested LSH inner map
is_large    = np.zeros(NUM_BUCKETS, dtype=np.int32)
bucket_size = outer_counts.copy()

# Map original bucket_indexes keys -> bucket IDs
orig_key_to_bid = {}
for key in bucket_indexes.keys():
    band_idx = key[0]
    if len(key) == 2:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1)
    elif len(key) == 4:
        v0, v1 = int(key[1][0]), int(key[1][1])
        hk = (band_idx, v0, v1, key[3])
    else:
        hk = key
    if hk in bucket_key_to_id:
        orig_key_to_bid[key] = bucket_key_to_id[hk]

for key, meta in bucket_indexes.items():
    if meta['use_index']:  # removed index_type check — cache has None
        bid = orig_key_to_bid.get(key)
        if bid is not None:
            is_large[bid] = 1

n_large = int(is_large.sum())
n_small = NUM_BUCKETS - n_large
print(f"  Large buckets (nested LSH): {n_large:,}")
print(f"  Small buckets (exact scan): {n_small:,}")
print(f"  Metadata built ({time.time()-t0:.2f}s)")

# ---- Step 4: Build inner CSR for large buckets ----
print("Step 4: Building inner CSR for large buckets...")
t0 = time.time()

# inner_key_to_id: (bid, inner_band_idx, iv0, iv1) -> inner_bucket_id
inner_key_to_id   = {}
inner_bucket_list = []  # inner_bucket_id -> (bid, inner_band_idx, iv0, iv1)

# First pass: collect all unique inner bucket keys
for key, meta in bucket_indexes.items():
    if not meta['use_index']:
        continue
    bid = orig_key_to_bid.get(key)
    if bid is None:
        continue
    for (ib, ikey), local_ids in meta['inner_map'].items():
        iv0, iv1 = int(ikey[0]), int(ikey[1])
        ik = (bid, ib, iv0, iv1)
        if ik not in inner_key_to_id:
            inner_key_to_id[ik] = len(inner_bucket_list)
            inner_bucket_list.append(ik)

NUM_INNER_BUCKETS = len(inner_key_to_id)

# Inner CSR: inner_bucket -> local member indices (local to their outer bucket)
inner_counts = np.zeros(NUM_INNER_BUCKETS, dtype=np.int32)
for key, meta in bucket_indexes.items():
    if not meta['use_index']:
        continue
    bid = orig_key_to_bid.get(key)
    if bid is None:
        continue
    for (ib, ikey), local_ids in meta['inner_map'].items():
        iv0, iv1 = int(ikey[0]), int(ikey[1])
        ik = (bid, ib, iv0, iv1)
        iid = inner_key_to_id[ik]
        inner_counts[iid] = len(local_ids)

inner_offsets_np = np.zeros(NUM_INNER_BUCKETS + 1, dtype=np.int32)
inner_offsets_np[1:] = np.cumsum(inner_counts)
total_inner = int(inner_offsets_np[-1])

inner_members_np = np.empty(total_inner, dtype=np.int32)
for key, meta in bucket_indexes.items():
    if not meta['use_index']:
        continue
    bid = orig_key_to_bid.get(key)
    if bid is None:
        continue
    for (ib, ikey), local_ids in meta['inner_map'].items():
        iv0, iv1 = int(ikey[0]), int(ikey[1])
        ik = (bid, ib, iv0, iv1)
        iid = inner_key_to_id[ik]
        s = inner_offsets_np[iid]
        e = inner_offsets_np[iid + 1]
        inner_members_np[s:e] = local_ids

print(f"  Inner CSR: {NUM_INNER_BUCKETS:,} inner buckets, {total_inner:,} assignments ({time.time()-t0:.2f}s)")

# ---- Step 5: Build band_key -> bucket_id lookup table ----
print("Step 5: Building query-time band lookup table...")
t0 = time.time()

# For each (band_idx, v0, v1) -> bucket_id
# Store as sorted arrays for binary search on GPU
# Format: for each band b, store sorted list of (hash_val, bucket_id)

# We build B separate lookup arrays (one per band)
# Each: sorted by hash(v0, v1), stores bucket_id
band_lookup    = []   # band_lookup[b] = (hash_vals_sorted, bucket_ids_sorted)
band_lookup_v0 = []   # sorted-aligned v0 (for exact verification on the CPU side)
band_lookup_v1 = []   # sorted-aligned v1

for b in range(B):
    hvals = []
    bids  = []
    v0s   = []
    v1s   = []
    for key in part_map.keys():
        if key[0] != b:
            continue
        if len(key) == 2:
            v0, v1 = int(key[1][0]), int(key[1][1])
            hk = (b, v0, v1)
        elif len(key) == 4:
            v0, v1 = int(key[1][0]), int(key[1][1])
            hk = (b, v0, v1, key[3])
        else:
            continue
        h = hash_band_key(b, v0, v1)
        hvals.append(h)
        bids.append(bucket_key_to_id[hk])
        v0s.append(v0)
        v1s.append(v1)

    if hvals:
        hvals = np.array(hvals, dtype=np.int64)
        bids  = np.array(bids,  dtype=np.int32)
        v0s   = np.array(v0s,   dtype=np.int64)
        v1s   = np.array(v1s,   dtype=np.int64)
        order = np.argsort(hvals)
        band_lookup.append((hvals[order], bids[order]))
        band_lookup_v0.append(v0s[order])
        band_lookup_v1.append(v1s[order])
    else:
        band_lookup.append((np.array([], dtype=np.int64),
                            np.array([], dtype=np.int32)))
        band_lookup_v0.append(np.array([], dtype=np.int64))
        band_lookup_v1.append(np.array([], dtype=np.int64))

print(f"  Band lookup built for {B} bands ({time.time()-t0:.2f}s)")

# ---- Step 6: Move everything to GPU ----
print("Step 6: Moving index to GPU...")
t0 = time.time()

gpu_outer_offsets = cp.asarray(outer_offsets)   # (NUM_BUCKETS+1,) int32
gpu_outer_members = cp.asarray(outer_members)   # (total_outer,)   int32
gpu_is_large      = cp.asarray(is_large)         # (NUM_BUCKETS,)   int32
gpu_bucket_size   = cp.asarray(bucket_size)      # (NUM_BUCKETS,)   int32
gpu_inner_offsets = cp.asarray(inner_offsets_np) # (NUM_INNER+1,)   int32
gpu_inner_members = cp.asarray(inner_members_np) # (total_inner,)   int32

# Band lookup arrays on GPU
gpu_band_hvals = [cp.asarray(band_lookup[b][0]) for b in range(B)]
gpu_band_bids  = [cp.asarray(band_lookup[b][1]) for b in range(B)]

# sig_all on GPU (int64 for exact band key matching)
gpu_sig_all = cp.asarray(sig_all.astype(np.int64))   # (N, 100) int64; WMH values are integer-valued floats

# inner bucket lookup: (bid, ib) -> range in inner CSR
# Build a flat array: for each large bucket bid,
# store offset into a per-bid inner bucket list
# We need: given (bid, ib, iv0, iv1) -> inner_bucket_id
# Store as: for each bid, sorted (hash(ib,iv0,iv1), inner_id)
large_bid_list = np.where(is_large)[0]
# For each large bucket, store its inner bucket hash+id arrays
inner_per_bid_hvals  = {}
inner_per_bid_iids   = {}

for key, meta in bucket_indexes.items():
    if not meta['use_index']:
        continue
    bid = orig_key_to_bid.get(key)
    if bid is None:
        continue
    hvals_b, iids_b = [], []
    for (ib, ikey), local_ids in meta['inner_map'].items():
        iv0, iv1 = int(ikey[0]), int(ikey[1])
        ik  = (bid, ib, iv0, iv1)
        iid = inner_key_to_id[ik]
        h   = hash_band_key(ib, iv0, iv1)
        hvals_b.append(h)
        iids_b.append(iid)
    hvals_b = np.array(hvals_b, dtype=np.int64)
    iids_b  = np.array(iids_b,  dtype=np.int32)
    order   = np.argsort(hvals_b)
    inner_per_bid_hvals[bid] = hvals_b[order]  # numpy; GPU move deferred to Cell 10
    inner_per_bid_iids[bid]  = iids_b[order]

print(f"  GPU transfer done ({time.time()-t0:.2f}s)")

# ---- Summary ----
vram_used = (
    gpu_outer_offsets.nbytes + gpu_outer_members.nbytes +
    gpu_is_large.nbytes + gpu_inner_offsets.nbytes +
    gpu_inner_members.nbytes + gpu_sig_all.nbytes +
    sum(v.nbytes for v in gpu_band_hvals) +
    sum(v.nbytes for v in gpu_band_bids)
) / 1e6

print(f"\n=== GPU CSR Index Built ===")
print(f"Outer CSR   : {NUM_BUCKETS:,} buckets, {total_outer:,} assignments")
print(f"Inner CSR   : {NUM_INNER_BUCKETS:,} inner buckets, {total_inner:,} assignments")
print(f"Large buckets: {n_large:,} | Small: {n_small:,}")
print(f"VRAM used   : {vram_used:.1f} MB")
print(f"VRAM free   : {cp.cuda.Device(0).mem_info[0]/1e6:.1f} MB")

Step 1: Assigning bucket IDs...
  Unique buckets: 207,902  (0.19s)
Step 2: Building outer CSR...
  Outer CSR: 207,902 buckets, 9,350,950 assignments (0.60s)
Step 3: Building bucket metadata...
  Large buckets (nested LSH): 2,171
  Small buckets (exact scan): 205,731
  Metadata built (0.28s)
Step 4: Building inner CSR for large buckets...
  Inner CSR: 1,951,126 inner buckets, 102,987,420 assignments (6.91s)
Step 5: Building query-time band lookup table...


/tmp/ipykernel_2311492/2013012070.py:18: RuntimeWarning: overflow encountered in scalar multiply
  h = h ^ (np.int64(v0) * HASH_P2)
/tmp/ipykernel_2311492/2013012070.py:19: RuntimeWarning: overflow encountered in scalar multiply
  h = h ^ (np.int64(v1) * HASH_P3)


  Band lookup built for 50 bands (1.84s)
Step 6: Moving index to GPU...
  GPU transfer done (10.52s)

=== GPU CSR Index Built ===
Outer CSR   : 207,902 buckets, 9,350,950 assignments
Inner CSR   : 1,951,126 inner buckets, 102,987,420 assignments
Large buckets: 2,171 | Small: 205,731
VRAM used   : 648.3 MB
VRAM free   : 73488.5 MB


In [9]:
# ---- Move encoding arrays to GPU ----
print("Moving encoding arrays to GPU...")
gpu_ids_flat  = cp.asarray(ids_flat)
gpu_vals_flat = cp.asarray(vals_flat)
gpu_offsets   = cp.asarray(offsets)
print(f"  ids_flat  : {gpu_ids_flat.nbytes/1e9:.2f} GB")
print(f"  vals_flat : {gpu_vals_flat.nbytes/1e9:.2f} GB")

# ---- Precompute per-vector value sums (Sd) for the WJ algebraic identity ----
# WJ = num / (Sq + Sd - num), where Sq/Sd are sum of values per vector.
# Compute once over the whole corpus via a robust cumsum (handles empty rows).
_csum = np.concatenate([[0.0], np.cumsum(vals_flat.astype(np.float64))])
db_sum_all = (_csum[offsets[1:]] - _csum[offsets[:-1]]).astype(np.float32)
gpu_db_sum = cp.asarray(db_sum_all)
del _csum
print(f"  db_sum    : {gpu_db_sum.nbytes/1e6:.1f} MB  (N={len(db_sum_all):,})")
print(f"  VRAM free : {cp.cuda.Device(0).mem_info[0]/1e9:.2f} GB")


Moving encoding arrays to GPU...
  ids_flat  : 4.59 GB
  vals_flat : 4.59 GB
  db_sum    : 0.9 MB  (N=233,773)
  VRAM free : 64.27 GB


In [10]:
# ============================================================
# Cell 3 — Full GPU Pipeline
# CPU: band key lookup → bucket IDs (2ms, no hashing needed)
# GPU: inner LSH probe + scoreboard + prune + batch rerank
# ============================================================

import cupyx

# ---- Precompute per-large-bucket: band_idx it belongs to ----
# So at query time we know which signature slice to use
print("Precomputing large bucket metadata...")

# For each large bucket (by bid), store:
#   - which outer band_idx it came from
#   - its member_ids (for global ID resolution)
#   - nested_L2, nested_R2 for inner banding

large_bid_to_bandidx = {}   # bid -> outer band_idx
large_bid_to_members = {}   # bid -> np.int32 member_ids (local index -> global poly_id)

for key, meta in bucket_indexes.items():
    if not meta['use_index']:
        continue
    bid = orig_key_to_bid.get(key)
    if bid is None:
        continue
    large_bid_to_bandidx[bid] = key[0]   # outer band_idx
    large_bid_to_members[bid] = meta['member_ids']  # shape (bucket_size,) int32

print(f"Large buckets precomputed: {len(large_bid_to_bandidx)}")


# ---- Build inner bucket lookup: (bid, inner_hash) -> inner_bucket_id ----
# For GPU binary search: per large bucket, sorted (inner_hash, inner_id) arrays
# Already built as inner_per_bid_hvals / inner_per_bid_iids
# Flatten into one big array with a bid-level offset table

print("Building flat inner lookup...")
t0 = time.time()

large_bids_sorted = sorted(large_bid_to_bandidx.keys())
n_large_bids      = len(large_bids_sorted)

# bid -> position in large_bids_sorted
large_bid_to_pos  = {bid: pos for pos, bid in enumerate(large_bids_sorted)}

# Per large-bucket inner lookup sizes
inner_lookup_sizes   = np.array([
    len(inner_per_bid_hvals[bid]) for bid in large_bids_sorted
], dtype=np.int32)

inner_lookup_offsets = np.zeros(n_large_bids + 1, dtype=np.int32)
inner_lookup_offsets[1:] = np.cumsum(inner_lookup_sizes)
total_inner_lookup   = int(inner_lookup_offsets[-1])

# Flat inner hash + inner_id arrays
flat_inner_hvals = np.empty(total_inner_lookup, dtype=np.int64)
flat_inner_iids  = np.empty(total_inner_lookup, dtype=np.int32)

for pos, bid in enumerate(large_bids_sorted):
    s = inner_lookup_offsets[pos]
    e = inner_lookup_offsets[pos + 1]
    flat_inner_hvals[s:e] = np.asarray(inner_per_bid_hvals[bid])
    flat_inner_iids[s:e]  = np.asarray(inner_per_bid_iids[bid])

# Member ID resolution: for each large bucket, map local_idx -> global poly_id
# Flatten all member arrays
member_array_sizes   = np.array([
    len(large_bid_to_members[bid]) for bid in large_bids_sorted
], dtype=np.int32)
member_array_offsets = np.zeros(n_large_bids + 1, dtype=np.int32)
member_array_offsets[1:] = np.cumsum(member_array_sizes)
total_members_flat   = int(member_array_offsets[-1])

flat_members = np.empty(total_members_flat, dtype=np.int32)
for pos, bid in enumerate(large_bids_sorted):
    s = member_array_offsets[pos]
    e = member_array_offsets[pos + 1]
    flat_members[s:e] = large_bid_to_members[bid]

# Move to GPU
gpu_flat_inner_hvals    = cp.asarray(flat_inner_hvals)
gpu_flat_inner_iids     = cp.asarray(flat_inner_iids)
gpu_inner_lookup_offsets= cp.asarray(inner_lookup_offsets)
gpu_flat_members        = cp.asarray(flat_members)
gpu_member_offsets      = cp.asarray(member_array_offsets)

print(f"  Flat inner lookup : {total_inner_lookup:,} entries")
print(f"  Flat members      : {total_members_flat:,} entries")
print(f"  Done ({time.time()-t0:.2f}s)")
print(f"  VRAM free         : {cp.cuda.Device(0).mem_info[0]/1e9:.2f} GB")

# ---- GPU kernel: inner LSH probe for large buckets ----
_INNER_LSH_KERNEL = cp.RawKernel(r'''
__device__ long long hash_inner(int ib, long long v0, long long v1) {
    long long h = (long long)ib * 2654435761LL;
    h = h ^ (v0 * 805459861LL);
    h = h ^ (v1 * 3474701543LL);
    return h & 0x7FFFFFFFFFFFFFFFLL;
}

__device__ int bsearch64(
    const long long* arr, int n, long long target
) {
    int lo = 0, hi = n - 1;
    while (lo <= hi) {
        int mid = (lo + hi) >> 1;
        if (arr[mid] == target) return mid;
        if (arr[mid] < target)  lo = mid + 1;
        else                    hi = mid - 1;
    }
    return -1;
}

extern "C" __global__
void gpu_inner_lsh_probe(
    // Query info
    const long long* __restrict__ sig_all,      // (N_total, L) int64
    const int*       __restrict__ query_ids,    // (Q,)
    const int Q,
    const int L,

    // Which large buckets to probe per query:
    // For query qi, probe large_bucket_ids[qb_offsets[qi]..qb_offsets[qi+1]]
    const int* __restrict__ large_bucket_ids,   // flat list of (pos in large_bids_sorted)
    const int* __restrict__ qb_offsets,         // (Q+1,)

    // Inner lookup (per large bucket, sorted by inner hash)
    const long long* __restrict__ flat_inner_hvals,
    const int*       __restrict__ flat_inner_iids,
    const int*       __restrict__ inner_lookup_offsets, // (n_large+1,)

    // Inner CSR (inner_bucket_id -> local member indices)
    const int* __restrict__ inner_offsets,      // (NUM_INNER+1,)
    const int* __restrict__ inner_members,      // (total_inner,)

    // Member resolution (pos -> global poly IDs)
    const int* __restrict__ flat_members,       // flat member arrays
    const int* __restrict__ member_offsets,     // (n_large+1,)

    // Nested LSH params
    const int nested_B2,
    const int nested_R2,

    // Scoreboard (Q, DATA_END)
    float* __restrict__ scoreboard,
    const int DATA_END
) {
    int qi  = blockIdx.x;
    if (qi >= Q) return;

    int qid = query_ids[qi];
    int tid = threadIdx.x;

    int n_probe = qb_offsets[qi + 1] - qb_offsets[qi];
    if (n_probe == 0) return;

    // Each thread handles one large bucket
    for (int pi = tid; pi < n_probe; pi += blockDim.x) {
        int pos = large_bucket_ids[qb_offsets[qi] + pi];

        // Inner lookup range for this bucket
        int il_start = inner_lookup_offsets[pos];
        int il_size  = inner_lookup_offsets[pos + 1] - il_start;

        // Member offset for this bucket
        int mem_start = member_offsets[pos];

        // Probe all nested_B2 inner bands
        for (int ib = 0; ib < nested_B2; ib++) {
            // Get inner band key from query signature
            int sig_base = qid * L + ib * nested_R2;
            long long iv0 = sig_all[sig_base];
            long long iv1 = sig_all[sig_base + 1];

            long long ih = hash_inner(ib, iv0, iv1);

            // Binary search in this bucket's inner lookup
            const long long* il_hvals = flat_inner_hvals + il_start;
            const int*       il_iids  = flat_inner_iids  + il_start;

            int ipos = bsearch64(il_hvals, il_size, ih);
            if (ipos < 0) continue;

            int inner_id = il_iids[ipos];

            // Get local member indices from inner CSR
            int ms = inner_offsets[inner_id];
            int me = inner_offsets[inner_id + 1];

            // Resolve local -> global and scatter-add to scoreboard
            for (int m = ms; m < me; m++) {
                int local_idx = inner_members[m];
                int global_id = flat_members[mem_start + local_idx];
                if (global_id < DATA_END) {
                    atomicAdd(&scoreboard[qi * DATA_END + global_id], 1.0f);
                }
            }
        }
    }
}
''', 'gpu_inner_lsh_probe')

print("Inner LSH kernel compiled.")

_SMALL_PROBE_KERNEL = cp.RawKernel(r'''
extern "C" __global__
void gpu_small_probe(
    const int* __restrict__ sb_starts,   // (n_pairs,) start in outer_members
    const int* __restrict__ sb_sizes,    // (n_pairs,) bucket size
    const int* __restrict__ sb_qidxs,   // (n_pairs,) which query
    const int* __restrict__ outer_members,
    float*     __restrict__ scoreboard,
    const int  DATA_END,
    const int  n_pairs
) {
    // One block per (query, bucket) pair
    int pair_idx = blockIdx.x;
    if (pair_idx >= n_pairs) return;

    int qi      = sb_qidxs[pair_idx];
    int ms      = sb_starts[pair_idx];
    int bk_size = sb_sizes[pair_idx];

    // Threads within block share the work for this bucket
    for (int t = threadIdx.x; t < bk_size; t += blockDim.x) {
        int poly_id = outer_members[ms + t];
        if (poly_id < DATA_END) {
            atomicAdd(&scoreboard[qi * DATA_END + poly_id], 1.0f);
        }
    }
}
''', 'gpu_small_probe')

print("Small probe kernel compiled.")


# ---- CPU band key lookup (fast — replaces hashing) ----
def cpu_get_bucket_ids_for_query(query_sig):
    small_bids = []
    large_poss = []

    # Access sig as raw numpy — avoid repeated attribute lookups
    sig_np = query_sig if isinstance(query_sig, np.ndarray) else np.asarray(query_sig)

    for b in range(min(B, 40)):
        # Inline band key extraction — faster than banding.get_band()
        s = b * R
        band_key  = (int(sig_np[s]), int(sig_np[s + 1]))
        exact_key = (b, band_key)

        bid = orig_key_to_bid.get(exact_key)
        if bid is not None:
            if is_large[bid]:
                pos = large_bid_to_pos.get(bid)
                if pos is not None:
                    large_poss.append(pos)
            else:
                small_bids.append(bid)

        # Split buckets
        split_prefix = (b, band_key, 'SPLIT')
        sl = split_index.get(split_prefix, [])
        for sk in sl:
            bid = orig_key_to_bid.get(sk)
            if bid is not None:
                if is_large[bid]:
                    pos = large_bid_to_pos.get(bid)
                    if pos is not None:
                        large_poss.append(pos)
                else:
                    small_bids.append(bid)

    return small_bids, large_poss

# ================================
# Kernel 3: Batch Weighted Jaccard rerank
# ================================
# Dense-query WJ kernel: one block per query, query densified once into a
# global scratch slice, each candidate thread scans only its own nnz with
# branchless dense lookups. Uses the identity den = Sq + Sd - num.
_WJ_BATCH_KERNEL = cp.RawKernel(r'''
extern "C" __global__
void batch_wj_dense(
    const int*   __restrict__ cand_ids,    // flat candidates, grouped by query
    const int*   __restrict__ q_offsets,   // (Q+1,) offsets into cand_ids
    const int*   __restrict__ query_ids,   // (Q,) global qid (indexes all_offsets)
    const int*   __restrict__ all_ids,
    const float* __restrict__ all_vals,
    const long*  __restrict__ all_offsets,
    const float* __restrict__ db_sum,      // (N,) precomputed sum of vals per vector
    float*       __restrict__ scratch,     // (gridDim.x * D) dense workspace, zeroed
    float*       __restrict__ out_scores,  // (total_pairs,)
    const int    Q,
    const int    D
) {
    float* dq = scratch + (long)blockIdx.x * (long)D;

    for (int qi = blockIdx.x; qi < Q; qi += gridDim.x) {
        int qid     = query_ids[qi];
        long q_s    = all_offsets[qid];
        long q_e    = all_offsets[qid + 1];
        int  q_nnz  = (int)(q_e - q_s);
        const int*   q_ids  = all_ids  + q_s;
        const float* q_vals = all_vals + q_s;

        // densify query into this block's scratch slice
        for (int i = threadIdx.x; i < q_nnz; i += blockDim.x)
            dq[q_ids[i]] = q_vals[i];
        __syncthreads();

        float Sq      = db_sum[qid];
        int   c_start = q_offsets[qi];
        int   n_cand  = q_offsets[qi + 1] - c_start;

        for (int c = threadIdx.x; c < n_cand; c += blockDim.x) {
            int  poly    = cand_ids[c_start + c];
            long db_s    = all_offsets[poly];
            long db_e    = all_offsets[poly + 1];
            int  db_nnz  = (int)(db_e - db_s);
            const int*   db_ids  = all_ids  + db_s;
            const float* db_vals = all_vals + db_s;

            float num = 0.0f;
            for (int k = 0; k < db_nnz; k++) {
                float qv = dq[db_ids[k]];   // 0 if query absent at this coord
                float dv = db_vals[k];
                num += (qv < dv) ? qv : dv; // min(qv, dv)
            }
            float den = Sq + db_sum[poly] - num;
            out_scores[c_start + c] = (den <= 0.0f) ? 1.0f : num / den;
        }
        __syncthreads();

        // reset only the touched entries so the slice stays zero for next query
        for (int i = threadIdx.x; i < q_nnz; i += blockDim.x)
            dq[q_ids[i]] = 0.0f;
        __syncthreads();
    }
}
''', 'batch_wj_dense')

# Dense-query scratch: one D-sized float slice per block (zero-initialized once).
WJ_NUM_BLOCKS  = 512
WJ_BLOCK_DIM   = 256
gpu_wj_scratch = cp.zeros(WJ_NUM_BLOCKS * int(D), dtype=cp.float32)
print(f"WJ dense scratch: {gpu_wj_scratch.nbytes/1e6:.0f} MB "
      f"({WJ_NUM_BLOCKS} blocks x D={D})")


def gpu_rerank_batch(query_ids, probe_results, topK=500):
    """Batch Weighted Jaccard rerank — all queries in one kernel launch.
    Returns (results, sub_timing) where sub_timing splits prep/kernel/post."""
    if len(query_ids) == 0:
        return {}, {"rk_prep_ms": 0.0, "rk_kernel_ms": 0.0, "rk_post_ms": 0.0}

    _t = time.time()
    Q = len(query_ids)
    cand_arrays  = [probe_results.get(qid, np.empty(0, dtype=np.int32))
                    for qid in query_ids]
    cand_lengths = np.array([len(c) for c in cand_arrays], dtype=np.int32)
    q_offsets_np = np.zeros(Q + 1, dtype=np.int32)
    q_offsets_np[1:] = np.cumsum(cand_lengths)
    total_pairs  = int(cand_lengths.sum())

    if total_pairs == 0:
        return ({qid: [] for qid in query_ids},
                {"rk_prep_ms": (time.time()-_t)*1000, "rk_kernel_ms": 0.0, "rk_post_ms": 0.0})

    cand_flat_np  = np.concatenate(cand_arrays).astype(np.int32)
    query_ids_np  = np.asarray(query_ids, dtype=np.int32)

    cand_gpu      = cp.asarray(cand_flat_np)
    q_offsets_gpu = cp.asarray(q_offsets_np)
    query_ids_gpu = cp.asarray(query_ids_np)
    scores_gpu    = cp.empty(total_pairs, dtype=cp.float32)
    rk_prep_ms = (time.time() - _t) * 1000

    _t = time.time()
    _WJ_BATCH_KERNEL(
        (WJ_NUM_BLOCKS,), (WJ_BLOCK_DIM,),
        (cand_gpu, q_offsets_gpu, query_ids_gpu,
         gpu_ids_flat, gpu_vals_flat, gpu_offsets, gpu_db_sum,
         gpu_wj_scratch, scores_gpu, Q, int(D))
    )
    cp.cuda.Stream.null.synchronize()
    rk_kernel_ms = (time.time() - _t) * 1000

    _t = time.time()
    scores_cpu = scores_gpu.get()
    cands_cpu  = cand_gpu.get()
    results    = {}
    for qi, qid in enumerate(query_ids):
        s, e = int(q_offsets_np[qi]), int(q_offsets_np[qi+1])
        if s == e:
            results[qid] = []
            continue
        q_scores = scores_cpu[s:e]
        q_cands  = cands_cpu[s:e]
        if len(q_scores) > topK:
            top_idx  = np.argpartition(q_scores, -topK)[-topK:]
            q_scores = q_scores[top_idx]
            q_cands  = q_cands[top_idx]
        order = np.argsort(-q_scores)
        results[qid] = list(zip(q_cands[order].tolist(), q_scores[order].tolist()))
    rk_post_ms = (time.time() - _t) * 1000

    return results, {"rk_prep_ms": rk_prep_ms, "rk_kernel_ms": rk_kernel_ms, "rk_post_ms": rk_post_ms}

print("Batch rerank kernel compiled.")


# ---- Vectorized exact band lookup (replaces the per-query Python loop) ----
# Uses the sorted per-band (hash, bid, v0, v1) arrays. searchsorted finds the
# hash range, then (v0, v1) are verified exactly so results are identical to the
# dict-based cpu_get_bucket_ids_for_query — just fully vectorized over the batch.
_HP1 = np.int64(2654435761); _HP2 = np.int64(805459861); _HP3 = np.int64(3474701543)

def _hash_band_vec(b, v0, v1):
    h = np.int64(b) * _HP1
    h = h ^ (v0 * _HP2)
    h = h ^ (v1 * _HP3)
    return h & np.int64(0x7FFFFFFFFFFFFFFF)

# bid -> position in large_bids_sorted (vectorized map; -1 if small/absent)
large_pos_of_bid = np.full(NUM_BUCKETS, -1, dtype=np.int32)
for _bid, _pos in large_bid_to_pos.items():
    large_pos_of_bid[_bid] = _pos

_NB_PROBE = min(B, 40)

def band_lookup_batch(query_ids_np):
    """Return (small_bids, q_small_offsets, large_poss, q_large_offsets) for a
    batch of queries, all grouped by query (numpy int32 arrays)."""
    Q = len(query_ids_np)
    qsig = sig_all[query_ids_np]                      # (Q, L)
    q_acc = []; b_acc = []
    with np.errstate(over='ignore'):                  # int64 hash multiply wraps (intended)
        for b in range(_NB_PROBE):
            hv = band_lookup[b][0]
            if hv.size == 0:
                continue
            v0q = qsig[:, b * R].astype(np.int64)
            v1q = qsig[:, b * R + 1].astype(np.int64)
            hq  = _hash_band_vec(b, v0q, v1q)
            lo  = np.searchsorted(hv, hq, 'left')
            hi  = np.searchsorted(hv, hq, 'right')
            counts = hi - lo
            tot = int(counts.sum())
            if tot == 0:
                continue
            base = np.repeat(np.cumsum(counts) - counts, counts)
            gidx = np.repeat(lo, counts) + (np.arange(tot) - base)
            cb   = band_lookup[b][1][gidx]
            cq   = np.repeat(np.arange(Q, dtype=np.int64), counts)
            # exact verify (v0, v1) to drop hash collisions
            ok = ((band_lookup_v0[b][gidx] == np.repeat(v0q, counts)) &
                  (band_lookup_v1[b][gidx] == np.repeat(v1q, counts)))
            q_acc.append(cq[ok]); b_acc.append(cb[ok])

    if q_acc:
        qrows = np.concatenate(q_acc); bids = np.concatenate(b_acc)
    else:
        qrows = np.empty(0, np.int64); bids = np.empty(0, np.int32)

    is_lg = is_large[bids].astype(bool)

    # large buckets -> positions, grouped by query
    lq   = qrows[is_lg]; lpos = large_pos_of_bid[bids[is_lg]]
    ol   = np.argsort(lq, kind='stable'); lq = lq[ol]; lpos = lpos[ol]
    l_off = np.zeros(Q + 1, np.int32)
    l_off[1:] = np.cumsum(np.bincount(lq, minlength=Q))

    # small buckets -> bids, grouped by query
    sq   = qrows[~is_lg]; sbid = bids[~is_lg]
    os_  = np.argsort(sq, kind='stable'); sq = sq[os_]; sbid = sbid[os_]
    s_off = np.zeros(Q + 1, np.int32)
    s_off[1:] = np.cumsum(np.bincount(sq, minlength=Q))

    return sbid.astype(np.int32), s_off, lpos.astype(np.int32), l_off



# ---- v1.20.1: ALL probed bucket ids per query (no small/large split) ----
def band_lookup_batch_all(query_ids_np):
    """Return (all_bids, q_offsets): every probed bucket id grouped by query.
    Banding only — exact WJ later scores all their members."""
    Q = len(query_ids_np)
    qsig = sig_all[query_ids_np]
    q_acc = []; b_acc = []
    with np.errstate(over='ignore'):
        for b in range(_NB_PROBE):
            hv = band_lookup[b][0]
            if hv.size == 0:
                continue
            v0q = qsig[:, b * R].astype(np.int64)
            v1q = qsig[:, b * R + 1].astype(np.int64)
            hq  = _hash_band_vec(b, v0q, v1q)
            lo  = np.searchsorted(hv, hq, 'left')
            hi  = np.searchsorted(hv, hq, 'right')
            counts = hi - lo
            tot = int(counts.sum())
            if tot == 0:
                continue
            base = np.repeat(np.cumsum(counts) - counts, counts)
            gidx = np.repeat(lo, counts) + (np.arange(tot) - base)
            cb   = band_lookup[b][1][gidx]
            cq   = np.repeat(np.arange(Q, dtype=np.int64), counts)
            ok = ((band_lookup_v0[b][gidx] == np.repeat(v0q, counts)) &
                  (band_lookup_v1[b][gidx] == np.repeat(v1q, counts)))
            q_acc.append(cq[ok]); b_acc.append(cb[ok])
    if q_acc:
        qrows = np.concatenate(q_acc); bids = np.concatenate(b_acc)
    else:
        qrows = np.empty(0, np.int64); bids = np.empty(0, np.int32)
    order = np.argsort(qrows, kind='stable'); qrows = qrows[order]; bids = bids[order]
    q_off = np.zeros(Q + 1, np.int32)
    q_off[1:] = np.cumsum(np.bincount(qrows, minlength=Q))
    return bids.astype(np.int32), q_off


# ---- Full GPU batch query pipeline ----
def gpu_query_pipeline(query_ids_list,
                       topK=500,
                       alpha_mult=0.0,
                       epsilon=200000,
                       data_end=DATA_END,
                       show_progress=False):
    """v1.20.1: banding selects buckets -> EXACT WJ over ALL their members.
    No nested-LSH filter, no collision pruning (alpha=0). Candidates come from the
    shared GPU CSR via member_ids; exact WJ rerank ranks them -> top-K."""
    Q = len(query_ids_list)
    alpha = B * alpha_mult   # 0 -> keep every touched candidate

    # Step 1: band lookup -> ALL probed bucket ids per query
    t_lookup = time.time()
    qids_np = np.asarray(query_ids_list, dtype=np.int64)
    all_bids, q_bid_off = band_lookup_batch_all(qids_np)
    t_lookup = time.time() - t_lookup

    # Step 2: scoreboard
    scoreboard_gpu = cp.zeros((Q, data_end), dtype=cp.float32)

    # Step 3: scatter ALL probed-bucket members (small_probe over every bucket)
    t_small = time.time()
    if all_bids.size:
        sb_starts_np = outer_offsets[all_bids]
        sb_sizes_np  = outer_offsets[all_bids + 1] - sb_starts_np
        counts       = np.diff(q_bid_off)
        sb_qidxs_np  = np.repeat(np.arange(Q, dtype=np.int32), counts)
        n_pairs      = len(all_bids)
        gpu_sb_starts = cp.asarray(sb_starts_np.astype(np.int32))
        gpu_sb_sizes  = cp.asarray(sb_sizes_np.astype(np.int32))
        gpu_sb_qidxs  = cp.asarray(sb_qidxs_np)
        _SMALL_PROBE_KERNEL(
            (n_pairs,), (128,),
            (gpu_sb_starts, gpu_sb_sizes, gpu_sb_qidxs,
             gpu_outer_members, scoreboard_gpu, data_end, n_pairs))
        cp.cuda.Stream.null.synchronize()
    t_small = time.time() - t_small
    t_large = 0.0   # no inner-LSH probe in v1.20.1

    # Step 4: prune — alpha=0 + huge epsilon => keep ALL probed-bucket members (deduped)
    t_prune = time.time()
    if alpha > 0:
        scoreboard_gpu[scoreboard_gpu < alpha] = 0.0
    rows_gpu, cols_gpu = cp.nonzero(scoreboard_gpu)
    vals_gpu   = scoreboard_gpu[rows_gpu, cols_gpu]
    counts_gpu = cp.bincount(rows_gpu, minlength=Q)
    cols_cpu   = cp.asnumpy(cols_gpu).astype(np.int32)
    vals_cpu   = cp.asnumpy(vals_gpu)
    counts_cpu = cp.asnumpy(counts_gpu)
    row_offsets = np.zeros(Q + 1, dtype=np.int64)
    row_offsets[1:] = np.cumsum(counts_cpu)
    probe_results = {}
    for qi, qid in enumerate(query_ids_list):
        s = int(row_offsets[qi]); e = int(row_offsets[qi + 1])
        if s == e:
            probe_results[qid] = np.empty(0, dtype=np.int32)
            continue
        touched = cols_cpu[s:e]
        if (e - s) > epsilon:
            scores  = vals_cpu[s:e]
            top_idx = np.argpartition(scores, -epsilon)[-epsilon:]
            touched = touched[top_idx]
        probe_results[qid] = touched
    del scoreboard_gpu, rows_gpu, cols_gpu, vals_gpu, counts_gpu
    t_prune = time.time() - t_prune

    # Step 5: exact WJ rerank over the candidates (shared CSR, real encodings)
    t_rerank = time.time()
    results, rk_sub = gpu_rerank_batch(query_ids_list, probe_results, topK=topK)
    t_rerank = time.time() - t_rerank

    t_total = t_lookup + t_small + t_large + t_prune + t_rerank
    timing = {
        "lookup_ms" : t_lookup * 1000,
        "small_ms"  : t_small  * 1000,
        "large_ms"  : t_large  * 1000,
        "prune_ms"  : t_prune  * 1000,
        "rerank_ms" : t_rerank * 1000,
        "rk_prep_ms"  : rk_sub["rk_prep_ms"],
        "rk_kernel_ms": rk_sub["rk_kernel_ms"],
        "rk_post_ms"  : rk_sub["rk_post_ms"],
        "total_ms"  : t_total  * 1000,
        "qps"       : Q / t_total,
    }
    return results, probe_results, timing


Precomputing large bucket metadata...
Large buckets precomputed: 2171
Building flat inner lookup...
  Flat inner lookup : 1,951,126 entries
  Flat members      : 5,149,371 entries
  Done (0.05s)
  VRAM free         : 64.23 GB
Inner LSH kernel compiled.
Small probe kernel compiled.
WJ dense scratch: 37 MB (512 blocks x D=18219)
Batch rerank kernel compiled.


In [11]:
# Warmup
print("Warming up GPU kernels...")
_q_warm = list(range(QUERY_START, QUERY_START + 5))
_ = gpu_query_pipeline(_q_warm, topK=FAST_TOPK, alpha_mult=FAST_ALPHA_MULT,
                        epsilon=FAST_EPSILON)
cp.cuda.Stream.null.synchronize()
print("Warmup done.")

# Full evaluation — batched to avoid OOM on scoreboard
query_ids_all = list(range(QUERY_START, QUERY_END))
results_full  = {}
timing_accum  = {"lookup_ms":0,"small_ms":0,"large_ms":0,
                 "prune_ms":0,"rerank_ms":0,
                 "rk_prep_ms":0,"rk_kernel_ms":0,"rk_post_ms":0,
                 "total_ms":0}

t_wall = time.time()
from tqdm import tqdm
for i in tqdm(range(0, len(query_ids_all), EVAL_BATCH_SIZE), desc="GPU eval"):
    batch = query_ids_all[i : i + EVAL_BATCH_SIZE]
    r, _probe, t = gpu_query_pipeline(
        batch, topK=FAST_TOPK, alpha_mult=FAST_ALPHA_MULT, epsilon=FAST_EPSILON,
    )
    results_full.update(r)
    for k in timing_accum:
        timing_accum[k] += t[k]
    cp.cuda.Stream.null.synchronize()
    cp.get_default_memory_pool().free_all_blocks()

t_wall_total = time.time() - t_wall
timing_full  = timing_accum
timing_full["qps"]      = len(query_ids_all) / t_wall_total
timing_full["total_ms"] = t_wall_total * 1000
print(f"Done. QPS={timing_full['qps']:.1f}  wall={t_wall_total:.1f}s")


Warming up GPU kernels...
Warmup done.


GPU eval: 100%|██████████| 234/234 [09:47<00:00,  2.51s/it]

Done. QPS=79.6  wall=587.7s


In [12]:
print("\n=== Recall (after rerank) ===")
recall = bp.evaluate_results(
    results_full, gt_neighbors, gt_offsets, gt_qid_map, ks=(10, 50, 500),
)
for k, v in recall.items():
    print(f"  R@{k} : {v:.4f}" if v is not None else f"  R@{k} : N/A")
print(f"  QPS  : {timing_full['qps']:.2f}")

gt_dict = bp.read_ground_truth_list(GROUND_TRUTH_DIR)
print("\n=== Recall Buddhi formula ===")
buddhi_recall = bp.buddhi_recall_from_boi_results(results_full, gt_dict, K_list=(10, 50, 500))

total_ms = timing_full["total_ms"]
print("\n=== Timing breakdown (cumulative across all batches) ===")
rows = [
    ("Stage",             "ms total",  "% of total"),
    ("─"*22,              "─"*12,      "─"*10),
    ("CPU band lookup",   f"{timing_full['lookup_ms']:.0f}", f"{timing_full['lookup_ms']/total_ms*100:.1f}%"),
    ("GPU small probe",   f"{timing_full['small_ms']:.0f}",  f"{timing_full['small_ms']/total_ms*100:.1f}%"),
    ("GPU large probe",   f"{timing_full['large_ms']:.0f}",  f"{timing_full['large_ms']/total_ms*100:.1f}%"),
    ("CPU prune",         f"{timing_full['prune_ms']:.0f}",  f"{timing_full['prune_ms']/total_ms*100:.1f}%"),
    ("GPU WJ rerank",     f"{timing_full['rerank_ms']:.0f}", f"{timing_full['rerank_ms']/total_ms*100:.1f}%"),
    ("  ├ rerank prep",   f"{timing_full['rk_prep_ms']:.0f}",   f"{timing_full['rk_prep_ms']/total_ms*100:.1f}%"),
    ("  ├ rerank kernel", f"{timing_full['rk_kernel_ms']:.0f}", f"{timing_full['rk_kernel_ms']/total_ms*100:.1f}%"),
    ("  └ rerank post",   f"{timing_full['rk_post_ms']:.0f}",   f"{timing_full['rk_post_ms']/total_ms*100:.1f}%"),
    ("Wall total",        f"{total_ms:.0f}",                 "100.0%"),
]
for label, ms, pct in rows:
    print(f"  {label:<22} {ms:>12} {pct:>10}")



=== Recall (after rerank) ===
[evaluate_results] Skipped 2088 queries with no GT entry.
  R@10 : 0.9927
  R@50 : 0.9955
  R@500 : 0.9983
  QPS  : 79.56

=== Recall Buddhi formula ===
  R@10: full=0.9484  adj=0.9927
  R@50: full=0.9511  adj=0.9955
  R@500: full=0.9537  adj=0.9983

=== Timing breakdown (cumulative across all batches) ===
  Stage                      ms total % of total
  ────────────────────── ──────────── ──────────
  CPU band lookup                5686       1.0%
  GPU small probe                2183       0.4%
  GPU large probe                   0       0.0%
  CPU prune                     14046       2.4%
  GPU WJ rerank                564680      96.1%
    ├ rerank prep                4644       0.8%
    ├ rerank kernel            540476      92.0%
    └ rerank post               19546       3.3%
  Wall total                   587663     100.0%


In [13]:
bp.log_result(
    notebook="v1.20.1_GPU_WJ_buckets",
    backend="gpu_wj_buckets_exact",
    dataset=_dataset_key,
    n_db=int(DATA_END),
    n_query=int(QUERY_END - QUERY_START),
    D=int(D),
    L=int(L),
    B=int(B),
    R=int(R),
    tau_min_size=int(BUCKET_INDEX_MIN_SIZE),
    split_thresh=int(REB_SPLIT_LARGE),
    index_metric="-",
    backend_params=f"exactWJ_allbucketmembers_gpubatch{EVAL_BATCH_SIZE}",
    alpha_mult=FAST_ALPHA_MULT,
    epsilon=int(FAST_EPSILON),
    efS=0,
    b_max=MAX_BANDS_TO_PROBE,
    stage="rerank",
    r10=recall.get(10),
    r50=recall.get(50),
    r500=recall.get(500),
    qps=timing_full["qps"],
    build_time_sec=build_stats.get("build_time_sec"),
    source="v1.20.1",
)


[log_result] → /raid/ruban/boi_multi_index/results.csv
  v1.20.1_GPU_WJ_buckets | gpu_wj_buckets_exact | rerank | R@10=0.9927 R@50=0.9955 R@500=0.9983 | QPS=79.56


## Epsilon sweep
Trades WJ-rerank work (the dominant kernel cost) against recall. Lower epsilon = fewer candidates reranked = higher QPS. Pick the knee of the curve.

In [14]:
# ---- Epsilon sweep: recall vs QPS ----
import time
from tqdm import tqdm

def run_eval_for_epsilon(epsilon, batch_size=EVAL_BATCH_SIZE):
    query_ids_all = list(range(QUERY_START, QUERY_END))
    results = {}
    t0 = time.time()
    for i in range(0, len(query_ids_all), batch_size):
        batch = query_ids_all[i:i + batch_size]
        r, _, _ = gpu_query_pipeline(
            batch, topK=FAST_TOPK, alpha_mult=FAST_ALPHA_MULT, epsilon=epsilon,
        )
        results.update(r)
        cp.cuda.Stream.null.synchronize()
        cp.get_default_memory_pool().free_all_blocks()
    wall = time.time() - t0
    rec = bp.evaluate_results(results, gt_neighbors, gt_offsets, gt_qid_map, ks=(10, 50, 500))
    return rec, len(query_ids_all) / wall

EPS_GRID = [4000, 3000, 2000, 1500, 1000]

# warmup once
_ = gpu_query_pipeline(list(range(QUERY_START, QUERY_START + 5)),
                       topK=FAST_TOPK, alpha_mult=FAST_ALPHA_MULT, epsilon=EPS_GRID[0])
cp.cuda.Stream.null.synchronize()

sweep_rows = []
for eps in tqdm(EPS_GRID, desc="epsilon sweep"):
    rec, qps = run_eval_for_epsilon(eps)
    sweep_rows.append((eps, rec, qps))

print(f"\n{'epsilon':>8} | {'R@10':>7} {'R@50':>7} {'R@500':>7} | {'QPS':>7} | {'vs eps=' + str(EPS_GRID[0]):>10}")
print("-" * 62)
base_r500 = sweep_rows[0][1][500]
for eps, rec, qps in sweep_rows:
    d500 = rec[500] - base_r500
    print(f"{eps:>8} | {rec[10]:>7.4f} {rec[50]:>7.4f} {rec[500]:>7.4f} | {qps:>7.1f} | {d500:>+9.4f}")


epsilon sweep:   0%|          | 0/5 [00:00<?, ?it/s]

epsilon sweep:   0%|          | 0/5 [01:04<?, ?it/s]


KeyboardInterrupt: 